In [1]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [2]:
import pickle
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from XReason.xgbooster import XGBooster
from ucimlrepo import fetch_ucirepo
import numpy as np

In [3]:
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


### UCIML Dataset: Default of Credit Card Clients

In [4]:
# fetch dataset
default_of_credit_card_clients = fetch_ucirepo(id=350)

# data (as pandas dataframes)
X = default_of_credit_card_clients.data.features
y = default_of_credit_card_clients.data.targets

# metadata
print(default_of_credit_card_clients.metadata)

# variable information
print(default_of_credit_card_clients.variables)

X.columns = default_of_credit_card_clients.variables.description[1:24]

y.columns = ['DEFAULT']
y
X['DEFAULT'] = y
X.rename(columns={'PAY_0': 'PAY_1'}, inplace=True)
X.rename(columns=lambda x: x.upper(), inplace=True)
X.head()
# remove useless and incorrect information
print(f"Dataset size before:\t{X.shape[0]}")
X = X.drop(X[X['MARRIAGE']==0].index)
X = X.drop(X[X['EDUCATION']==0].index)
X = X.drop(X[X['EDUCATION']==5].index)
X = X.drop(X[X['EDUCATION']==6].index)
print(f"Dataset size after:\t{X.shape[0]}")

pay_features = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for p in pay_features:
    X.loc[X[p]<0, p] = -1
    X.loc[X[p]>=0, p] = X.loc[X[p]>=0, p] + 1
    X[p] = X[p].astype('int64')
X['GRAD_SCHOOL'] = (X['EDUCATION'] == 1).astype('category')
X['UNIVERSITY'] = (X['EDUCATION'] == 2).astype('category')
X['HIGH_SCHOOL'] = (X['EDUCATION'] == 3).astype('category')
X.drop('EDUCATION', axis=1, inplace=True)

X['MALE'] = (X['SEX'] == 1).astype('category')
X.drop('SEX', axis=1, inplace=True)

X['MARRIED'] = (X['MARRIAGE'] == 1).astype('category')
X.drop('MARRIAGE', axis=1, inplace=True)

X.head()
y = X['DEFAULT'].values
X = X.drop('DEFAULT', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14, stratify=y)
print('Training set shape: ', np.shape(X_train))
print(f'- Defaulters:\t {len(y_train[y_train==1])}')
print(f'- Non-defaulters: {len(y_train[y_train==0])}')
print('Test set shape: ', np.shape(X_test))
print(f'- Defaulters:\t {len(y_test[y_test==1])}')
print(f'- Non-defaulters: {len(y_test[y_test==0])}')

{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

/var/folders/rk/_jtfhwrj2tg8_7_zxvdz0cw00000gp/T/ipykernel_32346/3300532718.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['DEFAULT'] = y
/var/folders/rk/_jtfhwrj2tg8_7_zxvdz0cw00000gp/T/ipykernel_32346/3300532718.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.rename(columns={'PAY_0': 'PAY_1'}, inplace=True)
/var/folders/rk/_jtfhwrj2tg8_7_zxvdz0cw00000gp/T/ipykernel_32346/3300532718.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pand

In [5]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [6]:
with open("model.mod.pkl", "wb") as f:
    pickle.dump(model, f)

In [7]:
categorical_values_dict = {
    "PAY_1":[-1,1,2,3,4,5,6,7,8,9],
    "PAY_2":[-1,1,2,3,4,5,6,7,8,9],
    "PAY_3":[-1,1,2,3,4,5,6,7,8,9],
    "PAY_4":[-1,1,2,3,4,5,6,7,8,9],
    "PAY_5":[-1,1,2,3,4,5,6,7,8,9],
    "PAY_6":[-1,1,2,3,4,5,6,7,8,9],
    "MALE": [0,1],
    "MARRIED": [0,1],
    "GRAD_SCHOOL": [0,1],
    "UNIVERSITY" : [0,1],
    "HIGH_SCHOOL": [0,1],
}

In [8]:
X_train

description,LIMIT_BAL,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,GRAD_SCHOOL,UNIVERSITY,HIGH_SCHOOL,MALE,MARRIED
16550,220000,28,1,1,1,1,1,1,38074,43083,...,5000,20012,23099,10029,30019,False,True,False,True,True
27787,70000,24,1,1,1,1,1,1,67897,62826,...,2916,2004,15006,3000,2200,False,True,False,False,False
14555,170000,36,1,1,1,1,1,1,158651,143801,...,4707,4400,4450,4650,4500,False,True,False,False,True
3742,280000,34,-1,1,1,1,1,1,11584,12211,...,3000,0,123,5000,0,True,False,False,False,True
11250,150000,25,1,1,1,1,1,1,147476,140352,...,4100,4000,2675,2500,1500,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6101,230000,33,1,1,1,1,1,1,218094,225070,...,9100,4600,4200,4100,5000,True,False,False,True,True
14811,200000,36,-1,-1,-1,-1,-1,-1,390,1308,...,1662,14987,34902,390,7500,False,True,False,False,True
7935,80000,24,1,1,1,1,1,1,77139,78690,...,2100,2000,3000,2500,2100,False,True,False,False,False
21243,190000,48,1,-1,1,1,1,1,6450,189102,...,7779,6092,5018,6168,1218,False,True,False,True,True


In [9]:
xgb = XGBooster(
    from_model="model.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'developer',
)

loading model from  model.mod.pkl
loading datainfo from  model.mod.pkl.splitdata.pkl
loading data from  model.mod.pkl.splitdata.pkl


In [10]:
xgb.encode()

In [11]:
sample = X_test.iloc[7]

In [12]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Default Risk")

FFA Graph all: ./ffa_all.png
FFA Graph top5: ./ffa_top5.png


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [13]:
xgb = XGBooster(
    from_model="model.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'decision_maker',
)

loading model from  model.mod.pkl
loading datainfo from  model.mod.pkl.splitdata.pkl
loading data from  model.mod.pkl.splitdata.pkl


In [14]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Default Risk")


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [15]:
xgb = XGBooster(
    from_model="model.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'model_subject',
)

loading model from  model.mod.pkl
loading datainfo from  model.mod.pkl.splitdata.pkl
loading data from  model.mod.pkl.splitdata.pkl


In [16]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Default Risk")


Accordion(children=(Tab(children=(HTML(value='\n                <div class="box">\n                    <div cl…

### Dog Adoption dataset

In [17]:
df = pd.read_csv('../dog_data/dog_adoption_master.csv')

In [18]:
y = df['returned']
y_extra = df[['return_reason','days_to_return']]
X = df.drop(['returned','return_reason','days_to_return','adoption_id'], axis=1)

In [19]:

X['breed_group'] = X['breed_group'].astype('category')
# X['terrier'] = (X['breed_group'] == 'terrier').astype('category')
# X['herding'] = (X['breed_group'] == 'herding').astype('category')
# X['working'] = (X['breed_group'] == 'working').astype('category')
# X['hound'] = (X['breed_group'] == 'hound').astype('category')
# X['toy'] = (X['breed_group'] == 'toy').astype('category')
# X['sporting'] = (X['breed_group'] == 'sporting').astype('category')
# X['mixed'] = (X['breed_group'] == 'mixed').astype('category')
#
# X.drop('breed_group', axis=1, inplace=True)


X['size'] = X['size'].astype('category')
# X['small_dog'] = (X['size'] == 'small').astype('category')
# X['medium_dog'] = (X['size'] == 'medium').astype('category')
# X['large_dog'] = (X['size'] == 'large').astype('category')
# X['xlarge_dog'] = (X['size'] == 'xlarge').astype('category')
#
# X.drop('size', axis=1, inplace=True)

X['intake_type'] = X['intake_type'].astype('category')

# X['stray'] = (X['intake_type'] == 'stray').astype('category')
# X['owner_surrender'] = (X['intake_type'] == 'owner_surrender').astype('category')
# X['transfer'] = (X['intake_type'] == 'transfer').astype('category')
# X['born_in_care'] = (X['intake_type'] == 'born_in_care').astype('category')

# X.drop('intake_type', axis=1, inplace=True)


X['previously_returned'] = X['previously_returned'].astype('category')

X['medical_needs'] = X['medical_needs'].astype('category')
# X['no_medical_needs'] = (X['medical_needs'] == 'none').astype('category')
# X['minor_medical_needs'] = (X['medical_needs'] == 'minor').astype('category')
# X['chronic_medical_needs'] = (X['medical_needs'] == 'chronic').astype('category')
#
# X.drop('medical_needs', axis=1, inplace=True)

X['neutered'] = X['neutered'].astype('category')
#
# X['house_trained'] = X['house_trained'].astype('category')
# X['first_time_owner'] = X['first_time_owner'].astype('category')
# X['household_has_kids'] = X['household_has_kids'].astype('category')
# X['household_has_pets'] = X['household_has_pets'].astype('category')

X['home_type'] = X['home_type'].astype('category')
# X['owner_house_owned'] = (X['home_type'] == 'house_owned').astype('category')
# X['owner_house_rented'] = (X['home_type'] == 'house_rented').astype('category')
# X['owner_apartment'] = (X['home_type'] == 'apartment').astype('category')
# X.drop('home_type', axis=1, inplace=True)



X['has_yard'] = X['has_yard'].astype('category')
X['adoption_counseling'] = X['adoption_counseling'].astype('category')
X['met_resident_pets'] = X['met_resident_pets'].astype('category')



X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)
print('Training set shape: ', np.shape(X_train))
print(f'- Returned:\t {len(y_train[y_train==1])}')
print(f'- Kept: {len(y_train[y_train==0])}')
print('Test set shape: ', np.shape(X_test))
print(f'- Returned:\t {len(y_test[y_test==1])}')
print(f'- Kept: {len(y_test[y_test==0])}')

Training set shape:  (31500, 28)
- Returned:	 4747
- Kept: 26753
Test set shape:  (10500, 28)
- Returned:	 1550
- Kept: 8950


In [20]:
from xgboost import XGBClassifier

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)

mask = X_train.notna().all(axis=1)
X_train = X_train.loc[mask]
y_train = y_train[mask]

mask = X_test.notna().all(axis=1)
X_test = X_test.loc[mask]
y_test = y_test[mask]


# X_train_enc = pd.get_dummies(X_train, drop_first=False)
# X_test_enc = pd.get_dummies(X_test, drop_first=False)

model = XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.75,
    random_state=42,
    enable_categorical=True,  # categorical dtype removed, no native splits
)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.75, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [22]:
with open("dogmodel2.mod.pkl", "wb") as f:
    pickle.dump(model, f)

In [26]:
preds = model.predict(X_test)

In [28]:
preds.sum()

435

In [29]:
len(preds)

10500

In [32]:
import numpy as np
np.where(preds == 1)[0]

array([   40,    73,    78,    86,    87,    90,   120,   124,   170,
         178,   194,   212,   219,   220,   225,   244,   296,   310,
         313,   408,   420,   428,   432,   441,   455,   486,   518,
         567,   595,   620,   645,   659,   660,   691,   699,   707,
         709,   802,   803,   813,   854,   857,   902,   918,   924,
         938,   939,   977,   985,   987,   995,  1036,  1048,  1073,
        1079,  1103,  1139,  1160,  1169,  1246,  1248,  1250,  1251,
        1255,  1296,  1332,  1432,  1482,  1492,  1500,  1522,  1527,
        1546,  1571,  1573,  1621,  1667,  1676,  1684,  1704,  1762,
        1787,  1820,  1882,  1901,  1937,  1959,  1969,  1970,  1992,
        2028,  2033,  2058,  2061,  2063,  2065,  2076,  2103,  2107,
        2132,  2141,  2155,  2226,  2246,  2253,  2286,  2347,  2370,
        2426,  2427,  2431,  2436,  2439,  2520,  2539,  2567,  2687,
        2746,  2783,  2787,  2797,  2828,  2841,  2886,  2924,  2964,
        2970,  2990,

In [34]:
X_test.iloc[40]

age_years                          5.2
size                             large
weight_kg                         34.4
breed_group                      mixed
intake_type                      stray
days_in_shelter                     43
previously_returned                  0
medical_needs                     none
neutered                             1
aggression_score                   3.6
anxiety_separation                 6.4
reactivity_to_dogs                 3.2
energy_level                       6.3
training_level                     3.6
house_trained                        0
first_time_owner                     1
household_has_kids                   1
household_has_pets                   1
home_type                 house_rented
has_yard                             1
hours_alone_per_day                4.8
adopter_activity_level            10.0
visits_before_adoption               2
met_resident_pets                    1
adoption_counseling                  0
expectation_score        

In [36]:
y_test.iloc[40]

1

In [35]:
X_test.iloc[313]

age_years                       2.1
size                         medium
weight_kg                      26.3
breed_group                   mixed
intake_type                   stray
days_in_shelter                  41
previously_returned               0
medical_needs                 minor
neutered                          0
aggression_score                7.6
anxiety_separation              6.7
reactivity_to_dogs              6.7
energy_level                    6.2
training_level                  3.3
house_trained                     0
first_time_owner                  0
household_has_kids                0
household_has_pets                1
home_type                 apartment
has_yard                          0
hours_alone_per_day             3.9
adopter_activity_level          4.9
visits_before_adoption            1
met_resident_pets                 1
adoption_counseling               0
expectation_score               7.0
energy_mismatch                 1.3
size_home_mismatch          

In [23]:
categorical_values_dict = {
    "size":['small', 'medium', 'large', 'xlarge'],
    "breed_group":['terrier','herding','working','hound' , 'toy', 'sporting' , 'mixed'],
    "intake_type":['stray' ,'owner_surrender','transfer', 'born_in_care'],
    "previously_returned":[0,1],
    "medical_needs":['none' , 'minor' , 'chronic'],
    "neutered":[0,1],
    "house_trained": [0,1],
    "first_time_owner": [0,1],
    "household_has_kids": [0,1],
    "household_has_pets" : [0,1],
    "home_type": ['house_owned','house_rented','apartment'],
    "has_yard": [0,1],
    "adoption_counseling": [0,1],
    "met_resident_pets": [0,1]
}

In [24]:
xgb = XGBooster(
    from_model="dogmodel2.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'developer'
)

loading model from  dogmodel2.mod.pkl
loading datainfo from  dogmodel2.mod.pkl.splitdata.pkl
loading data from  dogmodel2.mod.pkl.splitdata.pkl


In [25]:
xgb.encode()

In [26]:
sample = X_test.iloc[7]

In [27]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Rescue Failed")


saving ffa graph
names_all ['neutered', 'medical_needs', 'intake_type', 'anxiety_separation', 'age_years', 'aggression_score', 'size', 'breed_group', 'weight_kg']
values_all [0.027777777777777776, 0.05714285714285714, 0.07936507936507936, 0.10714285714285714, 0.1126984126984127, 0.1349206349206349, 0.14444444444444443, 0.16825396825396824, 0.16825396825396824]
names_top5 ['age_years', 'aggression_score', 'size', 'breed_group', 'weight_kg']
values_top5 [0.1126984126984127, 0.1349206349206349, 0.14444444444444443, 0.16825396825396824, 0.16825396825396824]
FFA Graph all: ./ffa_all.png
FFA Graph top5: ./ffa_top5.png


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [28]:
xgb = XGBooster(
    from_model="dogmodel2.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'decision_maker'
)

loading model from  dogmodel2.mod.pkl
loading datainfo from  dogmodel2.mod.pkl.splitdata.pkl
loading data from  dogmodel2.mod.pkl.splitdata.pkl


In [29]:
xgb.encode()

In [30]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Rescue Failed")


computing ffa
6.0


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [31]:
xgb = XGBooster(
    from_model="dogmodel2.mod.pkl",
    categorical_values_dict=categorical_values_dict,
    stakeholder_name = 'model_subject'
)
xgb.encode()

loading model from  dogmodel2.mod.pkl
loading datainfo from  dogmodel2.mod.pkl.splitdata.pkl
loading data from  dogmodel2.mod.pkl.splitdata.pkl


In [32]:
xgb.explain(sample, in_jupyter=True,
            prediction_label="Rescue Failed")


computing ffa
1.0


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [33]:
xgb.explain(sample, in_jupyter=False,
            prediction_label="Rescue Failed")


computing ffa
1.0


{'explanations': {'abd': ['size',
   'weight_kg',
   'breed_group',
   'neutered',
   'aggression_score',
   'anxiety_separation'],
  'con': {'features': ['medical_needs', 'aggression_score'],
   'values': {'medical_needs': 'chronic', 'aggression_score': 5.19999981},
   'verified': False}},
 'explained_instance': 'IF age_years = 8.2 AND size = small AND weight_kg = 7.0 AND breed_group = hound AND intake_type = owner_surrender AND days_in_shelter = 23 AND previously_returned = 0 AND medical_needs = minor AND neutered = 1 AND aggression_score = 2.0 AND anxiety_separation = 0.4 AND reactivity_to_dogs = 0.0 AND energy_level = 5.0 AND training_level = 3.5 AND house_trained = 1 AND first_time_owner = 1 AND household_has_kids = 0 AND household_has_pets = 1 AND home_type = apartment AND has_yard = 0 AND hours_alone_per_day = 1.4 AND adopter_activity_level = 6.7 AND visits_before_adoption = 1 AND met_resident_pets = 1 AND adoption_counseling = 0 AND expectation_score = 8.7 AND energy_mismatch =

In [34]:
from FRAME import FRAME

/Users/lkiern/XAI_Framework/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Instructions for updating:
non-resource variables are not supported in the long term


In [35]:
frame_xr = FRAME(
    model_path="dogmodel2.mod.pkl",
    model_type="xgboost",
    categorical_values_dict=categorical_values_dict,
    stakeholder_type = 'developer',
    time_limit=120
)

loading model from  dogmodel2.mod.pkl
loading datainfo from  dogmodel2.mod.pkl.splitdata.pkl
loading data from  dogmodel2.mod.pkl.splitdata.pkl


In [36]:
frame_xr.explain(sample,
            prediction_label="Rescue Failed")

saving ffa graph
names_all ['neutered', 'medical_needs', 'intake_type', 'anxiety_separation', 'age_years', 'aggression_score', 'size', 'breed_group', 'weight_kg']
values_all [0.027777777777777776, 0.05714285714285714, 0.07936507936507936, 0.10714285714285714, 0.1126984126984127, 0.1349206349206349, 0.14444444444444443, 0.16825396825396824, 0.16825396825396824]
names_top5 ['age_years', 'aggression_score', 'size', 'breed_group', 'weight_kg']
values_top5 [0.1126984126984127, 0.1349206349206349, 0.14444444444444443, 0.16825396825396824, 0.16825396825396824]
FFA Graph all: ./ffa_all.png
FFA Graph top5: ./ffa_top5.png


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [37]:
X_test

,age_years,size,weight_kg,breed_group,intake_type,days_in_shelter,previously_returned,medical_needs,neutered,aggression_score,...,home_type,has_yard,hours_alone_per_day,adopter_activity_level,visits_before_adoption,met_resident_pets,adoption_counseling,expectation_score,energy_mismatch,size_home_mismatch
12514,2.3,large,27.2,hound,stray,82,0,none,1,5.6,...,apartment,0,2.0,4.1,2,1,0,2.7,0.7,1
13069,4.0,xlarge,40.4,mixed,stray,6,0,none,1,3.3,...,apartment,0,1.6,4.0,1,0,0,7.4,1.0,1
15808,1.0,medium,16.0,hound,transfer,72,1,none,1,4.2,...,apartment,0,2.8,6.1,1,0,1,8.7,2.0,0
8336,2.0,large,31.8,terrier,stray,21,0,none,1,2.1,...,apartment,0,0.3,8.4,2,0,1,6.2,2.3,1
9839,6.7,small,7.9,hound,stray,74,0,none,1,2.3,...,house_rented,1,5.7,6.7,1,0,0,7.6,3.3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17136,1.7,small,5.6,mixed,owner_surrender,15,0,none,1,4.4,...,apartment,0,11.7,3.4,2,1,0,7.9,4.3,0
41408,1.0,large,40.7,mixed,transfer,79,0,none,1,1.3,...,apartment,0,3.6,7.3,4,0,0,6.9,1.8,1
4514,3.6,large,25.9,mixed,transfer,7,0,minor,1,2.0,...,house_owned,0,13.7,6.8,4,0,0,5.4,0.4,0
4796,1.7,large,40.0,mixed,owner_surrender,40,0,minor,1,1.7,...,apartment,0,6.0,4.1,1,0,1,5.6,2.1,1


In [38]:
X_test.to_csv('./dog_data_xtest_nonohe.csv')